# Week 16: Agentic policy verification

This notebook runs the **actual `app.agent.verify` loop** with scripted model decisions and retrieval fixtures. It needs no Ollama, Qdrant, Docker, or API key. The fixtures make each decision visible and reproducible; they do not measure live model quality.

A fixed search-then-answer pipeline is insufficient here: the next action depends on whether the preceding search found usable policy evidence.

In [1]:
from pathlib import Path
import importlib.util
import json

candidates = [Path.cwd(), Path.cwd() / 'week16' / 'assignment-week16']
root = next((p for p in candidates if (p / 'source' / 'scripts' / 'evaluate_agent.py').exists()), None)
if root is None:
    raise FileNotFoundError('Run this notebook from its folder or the repository root.')
harness_path = root / 'source' / 'scripts' / 'evaluate_agent.py'
spec = importlib.util.spec_from_file_location('evaluate_agent', harness_path)
harness = importlib.util.module_from_spec(spec)
spec.loader.exec_module(harness)
agent = harness.load_agent()
print(f'Loaded real agent loop from: {root / "source/api/app/agent.py"}')
print(f'Maximum decisions per request: {agent.MAX_STEPS}')

Loaded real agent loop from: C:\Users\rubin\Desktop\AI fellow\week16\assignment-week16\source\api\app\agent.py
Maximum decisions per request: 5


## Search, inspect, search again, answer

The first search has no matching policy. The next decision refines the query; the agent answers only after receiving a source quote.

In [2]:
case = harness.CASES[0]
decisions = iter(case['decisions'])
async def decide(messages):
    context = json.loads(messages[1]['content'])
    action = next(decisions)
    print(f"Step context: {len(context['evidence'])} visible snippets; next action: {action['action']}")
    return action, 42, 'scripted-model'
async def search(query, top_k):
    print(f'Search: {query!r} (top_k={top_k})')
    return case['hits'].get(query, [])
result = await agent.verify(case['question'], decide=decide, search=search)
print(json.dumps(result.as_dict(), indent=2))
assert result.status == 'completed' and result.steps == 3
assert [t['action'] for t in result.trajectory] == ['search', 'search', 'answer']

Step context: 0 visible snippets; next action: search
Search: 'cancel order' (top_k=3)
Step context: 0 visible snippets; next action: search
Search: 'cancel shipped order' (top_k=3)
Step context: 1 visible snippets; next action: answer
{
  "status": "completed",
  "answer": "Shipped orders cannot be cancelled.",
  "citations": [
    {
      "source": "CANCEL.md",
      "quote": "Shipped orders cannot be cancelled."
    }
  ],
  "trajectory": [
    {
      "step": 1,
      "action": "search",
      "query": "cancel order",
      "matches": 0
    },
    {
      "step": 2,
      "action": "search",
      "query": "cancel shipped order",
      "matches": 1
    },
    {
      "step": 3,
      "action": "answer"
    }
  ],
  "tokens": 126,
  "steps": 3,
  "model": "scripted-model"
}


## Failure and citation guards

The next fixtures show an invented quote being rejected, a search timeout being recorded, and the five-step stopping condition. The `success` column means the **expected behavior** occurred, including safe refusal; `task_completed` excludes an incomplete response.

In [3]:
rows = [await harness.run_case(agent, case) for case in harness.CASES]
print(f"Task completion: {sum(r['task_completed'] for r in rows)}/{len(rows)}")
print(f"Expected behavior: {sum(r['success'] for r in rows)}/{len(rows)}")
print(f"Tool-call correctness: {sum(r['tool_call_correct'] for r in rows)}/{len(rows)}")
for row in rows:
    path = ' -> '.join(step['action'] for step in row['trajectory'])
    print(f"{row['case']:<23} status={row['status']:<19} steps={row['steps']} tokens={row['tokens']:<3} {path}")
assert all(r['success'] for r in rows)
assert any(step['action'] == 'search_error' for step in rows[3]['trajectory'])
assert any(step['action'] == 'invalid_answer' for step in rows[2]['trajectory'])

Task completion: 4/5
Expected behavior: 5/5
Tool-call correctness: 5/5
refine_search           status=completed           steps=3 tokens=126 search -> search -> answer
clarification           status=needs_clarification steps=1 tokens=42  clarify
invented_citation       status=needs_clarification steps=3 tokens=126 search -> invalid_answer -> clarify
injected_tool_failure   status=needs_clarification steps=2 tokens=84  search_error -> clarify
step_limit              status=incomplete          steps=5 tokens=210 search -> search -> search -> search -> search -> step_limit


## Run with live services

Start the stack as described in `README.md`, then call `POST /verify` with JSON such as `{'message': 'Can I cancel an order after it ships?'}`. Live behavior depends on the local model and indexed policy corpus. The token figures above are simulated at 42 tokens per decision; the live endpoint uses provider-reported token usage when available.